# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Baselines

## load data

In [3]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/stf_baselines'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'learning_rate', 'batch_size', 'patience', 'train_epochs', 'lradj']
metric_names = ['smape', 'mase', 'owa']

df = []
for exp_dir in exp_dirs:
    runned, metric_dir, setting_dir = exist_stf_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_pkl(os.path.join(metric_dir, 'metrics.pkl'))
    metric = inverse_stf_metrics(metric, metric_names)
    for k, v in metric.items():
        result = config[params]
        result.loc[:, ['metric']] = k
        for kk, vv in v.items():
            result.loc[:, [kk]] = vv
        df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'metric'], inplace=True)
df.head(4)

,model,learning_rate,batch_size,patience,train_epochs,lradj,metric,smape,mase,owa
24,Autoformer,0.001,16,3,10,type1,Average,16.272,2.282,1.197
22,Autoformer,0.001,16,3,10,type1,Monthly,18.127,1.581,1.371
23,Autoformer,0.001,16,3,10,type1,Others,6.928,4.868,1.497
21,Autoformer,0.001,16,3,10,type1,Quarterly,14.612,1.882,1.350


## analysis

In [5]:
min_mode = 'each'

df2 = df.copy()

if min_mode == 'group':
    mse_mean = df2.groupby(['model', 'metric', 'learning_rate'])['smape'].mean().reset_index()
    idx = mse_mean.groupby(['model'])['smape'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[['model', 'learning_rate']], on=['model', 'learning_rate'], how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'metric'])['smape'].idxmin()
    df2 = df2.loc[min_mse_idx]

# df2 = df2[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df2 = df2[['model', 'metric', 'smape', 'mase', 'owa']]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df2.to_csv(f'{save_root}/stf_baselines_params.csv', index=False)

metric_order = ['Yearly', 'Quarterly', 'Monthly', 'Others', 'Average']
df2['metric'] = pd.Categorical(df2['metric'], categories=metric_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'FreTS', 'iTransformer', 'MICN', 'DLinear', 'FEDformer', 'Autoformer']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'metric'], inplace=True)

df2 = df2.set_index(['metric', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2.columns.levels[0]:
    columns.append((model, 'smape'))
    columns.append((model, 'mase'))
    columns.append((model, 'owa'))
df2 = df2[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df2.to_excel(f'{save_root}/stf_baselines.xlsx')
df2

model     Fredformer                 FBM_L                 FreTS         \
               smape   mase    owa   smape   mase    owa   smape   mase   
metric                                                                    
Yearly        13.509  3.028  0.794  13.949  3.005  0.805  13.514  3.042   
Quarterly     10.140  1.185  0.893  10.678  1.290  0.955  10.390  1.222   
Monthly       12.696  0.931  0.878  13.358  1.018  0.942  13.004  0.978   
Others         4.848  3.230  1.019   4.872  3.281  1.030   5.426  3.640   
Average       11.879  1.590  0.854  12.448  1.652  0.891  12.115  1.644   

model            iTransformer                  MICN               DLinear  \
             owa        smape   mase    owa   smape   mase    owa   smape   
metric                                                                      
Yearly     0.796       13.597  3.105  0.807  14.688  3.429  0.881  14.316   
Quarterly  0.917       10.126  1.175  0.888  11.652  1.428  1.050  10.511   
Monthly    0.911       12.719  0.927  0.877  13.882  1.089  0.993  13.364   
Others     1.145        4.895  3.207  1.021   6.502  4.515  1.396   5.371   
Average    0.877       11.907  1.602  0.858  13.163  1.880  0.977  12.498   

model                   FEDformer               Autoformer                
            mase    owa     smape   mase    owa      smape   mase    owa  
metric                                                                    
Yearly     3.134  0.832    13.594  3.063  0.801     16.162  3.602  0.948  
Quarterly  1.243  0.931    10.636  1.251  0.939     14.612  1.882  1.350  
Monthly    1.006  0.936    13.993  1.064  0.985     18.127  1.581  1.371  
Others     3.850  1.172     4.844  3.255  1.023      6.928  4.868  1.497  
Average    1.695  0.904    12.638  1.678  0.905     16.272  2.282  1.197

# finetune

## load data

In [6]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/stf_finetune'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'batch_size', 'patience', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'auxi_type', 'train_epochs', 'lradj']
metric_names = ['smape', 'mase', 'owa']

df = []
for exp_dir in exp_dirs:
    runned, metric_dir, setting_dir = exist_stf_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_pkl(os.path.join(metric_dir, 'metrics.pkl'))
    metric = inverse_stf_metrics(metric, metric_names)
    for k, v in metric.items():
        result = config[params]
        result.loc[:, ['metric']] = k
        for kk, vv in v.items():
            result.loc[:, [kk]] = vv
        df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'metric'], inplace=True)
df.head(4)

,model,batch_size,patience,learning_rate,rec_lambda,auxi_lambda,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,auxi_type,train_epochs,lradj,metric,smape,mase,owa
4,FreTS,16,3,0.0001,0.8500,0.1500,T,1,0,0.995,MAE,pca,10,type1,Average,12.134,1.653,0.880
19,FreTS,16,3,0.0003,0.8500,0.1500,T,1,0,0.995,MAE,pca,10,type1,Average,12.133,1.649,0.878
44,FreTS,16,3,0.0002,0.9995,0.0005,T,1,0,0.995,MAE,pca,10,type1,Average,12.140,1.659,0.881
59,FreTS,16,3,0.0001,0.8500,0.1500,T,1,0,0.900,MAE,pca,10,type1,Average,12.142,1.656,0.881


## preprocess

In [7]:
df3 = df.copy()
df3 = df3[
    (df3.pca_dim == 'T') & \
    (df3.reinit == 1) & \
    (df3.use_weights == 0) & \
    (df3.auxi_loss == 'MAE') & \
    (df3.auxi_type == 'pca')
]

df3.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
df3 = df3[['model', 'metric', 'batch_size', 'patience', 'learning_rate', 'alpha', 'rank_ratio', 'train_epochs', 'lradj', 'smape', 'mase', 'owa']]

## analysis

In [8]:
min_mode = 'group'

df2 = df3.copy()

columns = ['model', 'batch_size', 'patience', 'learning_rate', 'alpha', 'rank_ratio', 'train_epochs', 'lradj']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['smape'].mean().reset_index()
    idx = mse_mean.groupby(['model'])['smape'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'metric'])['smape'].idxmin()
    df2 = df2.loc[min_mse_idx]

columns = ['model', 'metric', 'smape', 'mase', 'owa', 'batch_size', 'patience', 'learning_rate', 'alpha', 'rank_ratio', 'train_epochs', 'lradj']
df2 = df2[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df2.to_csv(f'{save_root}/stf_finetune_params.csv', index=False)

metric_order = ['Yearly', 'Quarterly', 'Monthly', 'Others', 'Average']
df2['metric'] = pd.Categorical(df2['metric'], categories=metric_order, ordered=True)

model_order = ['Fredformer', 'FreTS', 'iTransformer']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'metric'], inplace=True)

df2 = df2.set_index(['metric', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2.columns.levels[0]:
    columns.append((model, 'smape'))
    columns.append((model, 'mase'))
    columns.append((model, 'owa'))
df2 = df2[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df2.to_excel(f'{save_root}/stf_finetune.xlsx')
df2

model     Fredformer                 FreTS               iTransformer         \
               smape   mase    owa   smape   mase    owa        smape   mase   
metric                                                                         
Yearly        13.485  3.010  0.791  13.493  3.026  0.794       13.603  3.125   
Quarterly     10.105  1.180  0.889  10.411  1.242  0.925       10.076  1.168   
Monthly       12.649  0.930  0.875  13.010  0.980  0.912       12.704  0.926   
Others         4.852  3.274  1.027   5.321  3.675  1.139        4.909  3.218   
Average       11.841  1.585  0.851  12.113  1.648  0.877       11.891  1.605   

model             
             owa  
metric            
Yearly     0.809  
Quarterly  0.883  
Monthly    0.876  
Others     1.024  
Average    0.858

# Merge

## load data

In [12]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

baseline = pd.read_csv(f'{save_root}/stf_baselines_params.csv')
finetune = pd.read_csv(f'{save_root}/stf_finetune_params.csv')

base = baseline.copy()
base = base[base.model.isin(['Fredformer', 'iTransformer', 'FreTS', 'MICN', 'DLinear', 'FEDformer'])]

best = finetune.copy()
best = best[best.model.isin(['Fredformer'])]
best['model'] = 'PDF'
best = best[['model', 'metric', 'smape', 'mase', 'owa']]

df3 = pd.concat([base, best], ignore_index=True)
df3 = df3[['model', 'metric', 'smape', 'mase', 'owa']]


metric_order = ['Yearly', 'Quarterly', 'Monthly', 'Others', 'Average']
df3['metric'] = pd.Categorical(df3['metric'], categories=metric_order, ordered=True)

model_order = ['PDF', 'Fredformer', 'iTransformer', 'FreTS', 'MICN', 'DLinear', 'FEDformer']
df3['model'] = pd.Categorical(df3['model'], categories=model_order, ordered=True)
df3.sort_values(by=['model', 'metric'], inplace=True)

df3 = df3.set_index(['metric', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df3.columns.levels[0]:
    columns.append((model, 'smape'))
    columns.append((model, 'mase'))
    columns.append((model, 'owa'))
df3 = df3[columns]

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
df3.round(3).to_csv(f'{save_root}/short_term.csv', float_format='%.3f')
df3

model         PDF               Fredformer               iTransformer         \
            smape   mase    owa      smape   mase    owa        smape   mase   
metric                                                                         
Yearly     13.485  3.010  0.791     13.509  3.028  0.794       13.597  3.105   
Quarterly  10.105  1.180  0.889     10.140  1.185  0.893       10.126  1.175   
Monthly    12.649  0.930  0.875     12.696  0.931  0.878       12.719  0.927   
Others      4.852  3.274  1.027      4.848  3.230  1.019        4.895  3.207   
Average    11.841  1.585  0.851     11.879  1.590  0.854       11.907  1.602   

model              FreTS                  MICN               DLinear         \
             owa   smape   mase    owa   smape   mase    owa   smape   mase   
metric                                                                        
Yearly     0.807  13.514  3.042  0.796  14.688  3.429  0.881  14.316  3.134   
Quarterly  0.888  10.390  1.222  0.917  11.652  1.428  1.050  10.511  1.243   
Monthly    0.877  13.004  0.978  0.911  13.882  1.089  0.993  13.364  1.006   
Others     1.021   5.426  3.640  1.145   6.502  4.515  1.396   5.371  3.850   
Average    0.858  12.115  1.644  0.877  13.163  1.880  0.977  12.498  1.695   

model            FEDformer                
             owa     smape   mase    owa  
metric                                    
Yearly     0.832    13.594  3.063  0.801  
Quarterly  0.931    10.636  1.251  0.939  
Monthly    0.936    13.993  1.064  0.985  
Others     1.172     4.844  3.255  1.023  
Average    0.904    12.638  1.678  0.905